In [13]:
import requests
from bs4 import BeautifulSoup

service_key = '34cf2a75ed8a3fd4404f4da20bb50ee752fd728f079f1dcc0cee0813fe3a649d'

def getParmacy(service_key : str, page_no : int = 1):
	"""공공 데이터에서 약국 정보를 가져와서 딕셔너리로 반환하는 함수"""
  # End Point + 활용신청 상세기능정보 > 상세기능에 있는 주소
	url = 'https://apis.data.go.kr/B551182/pharmacyInfoService/getParmacyBasisList'
	params = {
		'ServiceKey' : service_key,
		'pageNo' : page_no
	}

	try:
		response = requests.get(url, params=params)
		
		response.raise_for_status()
		
		# 응답 결과가 xml 파일이어서
		res_xml = response.text
		
		soup = BeautifulSoup(res_xml, 'lxml-xml')
		
		# 약국들 정보
		items = soup.find_all('item')
		
		yadmNms = []
		clCdNms = []
		sidoCdNms = []
		sgguCdNms = []
		addrs = []
		
		for item in items:
			yadmNms.append(item.find('yadmNm').text) # 병원명
			clCdNms.append(item.find('clCdNm').text) # 종별 코드명(약국)
			sidoCdNms.append(item.find('sidoCdNm').text) # 시도명
			sgguCdNms.append(item.find('sgguCdNm').text) # 시군구명
			addrs.append(item.find('addr').text) # 주소
			
		res_dic = {
			'병원명' : yadmNms,
			'종별코드명' : clCdNms,
			'시도명' : sidoCdNms,
			'시군구명' : sgguCdNms,
			'주소' : addrs
		}
		return res_dic
		
	except Exception as e:
		print(f"예외 발생 {e}")
		return {}
		

In [ ]:
import pandas as pd
import time

df = pd.DataFrame({
  '병원명' : [],
	'종별코드명' : [],
	'시도명' : [],
	'시군구명' : [],
	'주소' : []
	})

for i in range(1, 6):
  tmp_df = pd.DataFrame(getParmacy(service_key, i))
  df = pd.concat([df, tmp_df])
  print(f"{i} 페이지가 로딩되었습니다.")
  time.sleep(1)
  
print(df)

In [ ]:
# 인덱스 번호 초기화
df = df.reset_index(drop=True)

In [16]:
# csv 파일로 저장
df.to_csv('약국 목록.csv', index=False, encoding='utf-8-sig')

In [ ]:
# csv 파일 읽어오기
df2 = pd.read_csv('약국 목록.csv', encoding='utf-8')
print(df2)

In [21]:
# 경기에 있는 약국 수를 조회
print(f"경기에 있는 약국 수 : {len(df.loc[df['시도명'] == '경기'])}")

# 결측치가 있는지 확인
print(df.isnull().sum())

경기에 있는 약국 수 : 6
병원명      0
종별코드명    0
시도명      0
시군구명     0
주소       0
dtype: int64
